# Evidence Verification — The Crescent

Every quantitative claim in the stakeholder brief should be reproducible by running a cell in this notebook. If somebody asks *"where does the 501 come from?"*, the answer is a section number here.

**How to read this:**
1. Each section starts with the **biggest takeaway** — what the data says, in plain English.
2. A short code cell produces the table.
3. A `[↓ View SQL]` link jumps to the **SQL Queries Reference** at the bottom, where every query is shown formatted.

**Conventions:**
- Time-gate: every query filters `shows.date <= '2026-05-18'` (the analysis date the matrix was compiled on, pinned so numbers reproduce exactly).
- Dispute definition: `settlement.status IN ('disputed','revised','finalized') OR disputed_at IS NOT NULL`.

**Setup is one cell below. Skip it — every query in this notebook is rendered formatted at the bottom in [SQL Queries Reference](#sql-reference).**

In [1]:
# === SETUP ===
# Imports, DB connection, dates, helpers, and every SQL string used below.
# All queries are also rendered as formatted code blocks at the bottom of the
# notebook in the SQL Queries Reference section — that's the readable copy.

import re
import sqlite3
from pathlib import Path
from collections import Counter
import pandas as pd
from IPython.display import Markdown

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "Pallavi's" else Path.cwd()
DB_PATH = REPO_ROOT / "data" / "greenroom.db"
assert DB_PATH.exists(), f"DB not found at {DB_PATH}"

ANALYSIS_DATE = "2026-05-18"
TODAY = "2026-05-19"

conn = sqlite3.connect(DB_PATH)

def q(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params or {})

def inspect(show_id):
    """Pull every DB field for one show: deal + settlement. Verification helper."""
    return q("""
        SELECT d.show_id, d.deal_type, d.guarantee_amount, d.percentage,
               d.percentage_basis, d.expense_cap, d.hospitality_cap,
               d.deal_notes_freetext,
               st.status, st.gross_box_office, st.net_box_office,
               st.total_expenses, st.total_to_artist,
               st.calculation_json, st.signoff_text, st.notes AS settlement_notes
          FROM deals d
          JOIN settlements st ON d.show_id = st.show_id
         WHERE d.show_id = :sid
    """, {"sid": show_id}).T

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 60)

# --- SQL strings (the readable copy lives at the bottom of the notebook) ---

OVERVIEW_SQL = """
SELECT 'Time horizon (past shows)'         AS what,
       MIN(date) || '  to  ' || MAX(date)  AS value
  FROM shows WHERE date <= :gate
UNION ALL SELECT 'Past shows',
       CAST(COUNT(*) AS TEXT) FROM shows WHERE date <= :gate
UNION ALL SELECT 'Future shows (booked, not yet happened)',
       CAST(COUNT(*) AS TEXT) FROM shows WHERE date > :gate
UNION ALL SELECT 'Venue (single)',
       (SELECT name FROM venues LIMIT 1)
UNION ALL SELECT 'Distinct artists (past shows)',
       CAST(COUNT(DISTINCT artist_id) AS TEXT) FROM shows WHERE date <= :gate
UNION ALL SELECT 'Distinct agents (past shows)',
       CAST(COUNT(DISTINCT a.id) AS TEXT)
  FROM shows s
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  a  ON ar.agent_id = a.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Distinct agencies (past shows)',
       CAST(COUNT(DISTINCT a.agency_id) AS TEXT)
  FROM shows s
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  a  ON ar.agent_id = a.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Deals (one per show)',
       CAST(COUNT(*) AS TEXT) FROM deals d
  JOIN shows s ON d.show_id = s.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Settlements (one per show)',
       CAST(COUNT(*) AS TEXT) FROM settlements st
  JOIN shows s ON st.show_id = s.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Expense line items',
       CAST(COUNT(*) AS TEXT) FROM expenses e
  JOIN shows s ON e.show_id = s.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Comp line items',
       CAST(COUNT(*) AS TEXT) FROM comps c
  JOIN shows s ON c.show_id = s.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Ticket-sales rows (one aggregate per show)',
       CAST(COUNT(*) AS TEXT) FROM ticket_sales t
  JOIN shows s ON t.show_id = s.id
 WHERE s.date <= :gate
"""

DEALMIX_SQL = """
SELECT d.deal_type,
       COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) /
             (SELECT COUNT(*) FROM shows WHERE date <= :gate), 1) AS pct_of_501,
       SUM(CASE
             WHEN d.deal_type = 'flat'
                  AND ABS(st.total_to_artist - d.guarantee_amount) < 0.5
                  THEN 1
             WHEN d.deal_type = 'percentage_of_gross'
                  AND ABS(st.total_to_artist - (st.gross_box_office * d.percentage)) < 1
                  THEN 1
             ELSE 0
           END) AS formula_match
  FROM deals d
  JOIN shows s ON d.show_id = s.id
  JOIN settlements st ON st.show_id = s.id
 WHERE s.date <= :gate
 GROUP BY d.deal_type
 ORDER BY n DESC
"""

DISPUTED_DRILL_SQL = """
SELECT s.id AS show_id,
       s.date,
       d.deal_type,
       ag.name AS agent,
       agcy.name AS agency,
       d.guarantee_amount AS guar,
       st.total_to_artist AS payout,
       CASE WHEN st.recoups_json LIKE '%"status":"disputed"%'
            THEN 'Y' ELSE '.' END AS recoup_disputed,
       substr(st.signoff_text, 1, 60) AS signoff_snippet
  FROM settlements st
  JOIN shows s   ON st.show_id  = s.id
  JOIN deals d   ON d.show_id   = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id
 WHERE s.date <= :gate
   AND st.status = 'disputed'
 ORDER BY s.id
"""

DISPUTED_BY_DEALTYPE_SQL = """
SELECT d.deal_type,
       COUNT(*) AS n_disputed
  FROM settlements st
  JOIN shows s ON st.show_id = s.id
  JOIN deals d ON d.show_id  = s.id
 WHERE s.date <= :gate
   AND st.status = 'disputed'
 GROUP BY d.deal_type
 ORDER BY n_disputed DESC
"""

DISPUTED_BY_AGENT_SQL = """
SELECT ag.name AS agent,
       agcy.name AS agency,
       COUNT(*) AS n_disputed
  FROM settlements st
  JOIN shows s ON st.show_id = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id
 WHERE s.date <= :gate
   AND st.status = 'disputed'
 GROUP BY ag.id
 ORDER BY n_disputed DESC, agent
"""

RECOUP_DISPUTED_DRILL_SQL = """
SELECT s.id AS show_id,
       s.date,
       st.status AS settlement_status,
       d.deal_type,
       ag.name AS agent,
       agcy.name AS agency,
       json_extract(je.value, '$.category') AS category,
       json_extract(je.value, '$.label')    AS label,
       json_extract(je.value, '$.amount')   AS amount,
       substr(st.signoff_text, 1, 60)       AS signoff_snippet
  FROM settlements st
  JOIN shows s   ON st.show_id  = s.id
  JOIN deals d   ON d.show_id   = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id,
  json_each(st.recoups_json) je
 WHERE s.date <= :gate
   AND json_extract(je.value, '$.status') = 'disputed'
 ORDER BY s.id, json_extract(je.value, '$.label')
"""

RECOUP_DISPUTED_BY_CATEGORY_SQL = """
SELECT json_extract(je.value, '$.category') AS category,
       COUNT(*) AS n_line_items,
       COUNT(DISTINCT st.show_id) AS n_settlements,
       SUM(json_extract(je.value, '$.amount')) AS total_disputed_amount
  FROM settlements st
  JOIN shows s ON st.show_id = s.id,
  json_each(st.recoups_json) je
 WHERE s.date <= :gate
   AND json_extract(je.value, '$.status') = 'disputed'
 GROUP BY category
 ORDER BY n_line_items DESC
"""

DISPUTE_UNIVERSE_CTE = """
WITH dispute_universe AS (
  SELECT DISTINCT
         st.show_id,
         CASE WHEN st.status = 'disputed' THEN 1 ELSE 0 END AS is_ui_visible,
         CASE WHEN EXISTS (
                SELECT 1 FROM json_each(st.recoups_json) je
                 WHERE json_extract(je.value, '$.status') = 'disputed'
              ) THEN 1 ELSE 0 END AS has_hidden_recoup
    FROM settlements st
    JOIN shows s ON st.show_id = s.id
   WHERE s.date <= :gate
     AND ( st.status = 'disputed'
        OR EXISTS (
             SELECT 1 FROM json_each(st.recoups_json) je
              WHERE json_extract(je.value, '$.status') = 'disputed'
           )
         )
)
"""

DISPUTE_UNIVERSE_BY_AGENT_SQL = DISPUTE_UNIVERSE_CTE + """
SELECT ag.name AS agent,
       agcy.name AS agency,
       SUM(du.is_ui_visible) AS n_ui_visible,
       SUM(CASE WHEN du.is_ui_visible = 0 AND du.has_hidden_recoup = 1
                THEN 1 ELSE 0 END) AS n_hidden_only,
       COUNT(*) AS n_total
  FROM dispute_universe du
  JOIN shows s ON du.show_id = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id
 GROUP BY ag.id
 ORDER BY n_total DESC, agent
"""

DISPUTE_UNIVERSE_BY_AGENCY_SQL = DISPUTE_UNIVERSE_CTE + """
SELECT agcy.name AS agency,
       SUM(du.is_ui_visible) AS n_ui_visible,
       SUM(CASE WHEN du.is_ui_visible = 0 AND du.has_hidden_recoup = 1
                THEN 1 ELSE 0 END) AS n_hidden_only,
       COUNT(*) AS n_total,
       ROUND(100.0 * COUNT(*) / 42.0, 1) AS pct_of_42
  FROM dispute_universe du
  JOIN shows s ON du.show_id = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id
 GROUP BY agcy.id
 ORDER BY n_total DESC
"""

DEALS_FREETEXT_SQL = """
SELECT s.id AS show_id, d.deal_type, d.deal_notes_freetext
  FROM deals d JOIN shows s ON d.show_id = s.id
 WHERE s.date <= :gate
"""

VS_PATTERNS = [
    ("V8", "Vs + walkout pot",                            r"walkout pot"),
    ("V7", "Vs % of gross (with risk caveat)",            r"gross.*(?:Simpler math|riskier for venue)"),
    ("V6", "Vs % of GROSS, whichever greater",            r"vs\s+\d+%\s+of\s+GROSS"),
    ("V5", "Vs escalator/ratchet over capacity",          r"escalator|ratchets\s+to"),
    ("V9", "Vs inline capacity ratchet",                  r"\d+%\s+net\s+to\s+\d+%\s+sold"),
    ("V3", "Vs slash-split with walkout above breakeven", r"\d+/\d+\s+net,\s+walkout\s+above"),
    ("V2", "Vs slash-split (no walkout)",                 r"\d+/\d+\s+(?:net|after\s+expenses|split)"),
    ("V1", "Vs % of net, whichever greater",              r"of\s+net\s+after\s+expenses.*whichever\s+greater"),
    ("V4", "Vs % of net (compact)",                       r"g'tee\s+vs\s+\d+%\s+of\s+net"),
]

NON_VS_BUCKETS = {
    "percentage_of_net":   ("N1", "% of net, no guarantee"),
    "percentage_of_gross": ("G1", "% of gross"),
    "door":                ("D1", "Door deal"),
}

PATTERN_ORDER = ["V1","V2","V3","V4","V5","V6","V7","V8","V9","N1","G1","D1","Unmatched"]
PATTERN_DESCRIPTIONS = {pid: desc for pid, desc, _ in VS_PATTERNS}
PATTERN_DESCRIPTIONS.update({pid: desc for _, (pid, desc) in NON_VS_BUCKETS.items()})
PATTERN_DESCRIPTIONS["Unmatched"] = "(no pattern matched)"

def classify_pattern(deal_type, notes):
    """Classify a deal into one of 12 patterns + Unmatched. Flat returns None (excluded)."""
    if deal_type == "vs":
        for pid, _, regex in VS_PATTERNS:
            if re.search(regex, notes or ""):
                return pid
        return "Unmatched"
    if deal_type in NON_VS_BUCKETS:
        return NON_VS_BUCKETS[deal_type][0]
    return None  # flat — excluded

# §7: drilldown on every deal whose notes punt bonus terms to "see email thread."
# Single LIKE on deal_notes_freetext. Backs the "schema cannot hold what gets
# signed" argument: when bonuses are too complex, the source of truth moves
# outside Greenroom and the calculator has nothing to compute against.
SEE_EMAIL_THREAD_SQL = """
SELECT s.id AS show_id,
       s.date,
       d.deal_type,
       ag.name AS agent,
       agcy.name AS agency,
       st.status AS settlement_status,
       d.guarantee_amount AS guar,
       d.expense_cap,
       substr(d.deal_notes_freetext, 1, 130) AS deal_notes_snippet
  FROM deals d
  JOIN shows s ON d.show_id = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id
  JOIN settlements st ON st.show_id = s.id
 WHERE s.date <= :gate
   AND d.deal_notes_freetext LIKE '%see email thread%'
 ORDER BY s.id
"""

print("Setup loaded. Queries: OVERVIEW_SQL, DEALMIX_SQL, DISPUTED_DRILL_SQL, DISPUTED_BY_DEALTYPE_SQL, DISPUTED_BY_AGENT_SQL, RECOUP_DISPUTED_DRILL_SQL, RECOUP_DISPUTED_BY_CATEGORY_SQL, DISPUTE_UNIVERSE_BY_AGENT_SQL, DISPUTE_UNIVERSE_BY_AGENCY_SQL, DEALS_FREETEXT_SQL, SEE_EMAIL_THREAD_SQL.")
print("Helper: inspect('show_XXXX'). Classifier: classify_pattern(deal_type, notes).")

Setup loaded. Queries: OVERVIEW_SQL, DEALMIX_SQL, DISPUTED_DRILL_SQL, DISPUTED_BY_DEALTYPE_SQL, DISPUTED_BY_AGENT_SQL, RECOUP_DISPUTED_DRILL_SQL, RECOUP_DISPUTED_BY_CATEGORY_SQL, DISPUTE_UNIVERSE_BY_AGENT_SQL, DISPUTE_UNIVERSE_BY_AGENCY_SQL, DEALS_FREETEXT_SQL, SEE_EMAIL_THREAD_SQL.
Helper: inspect('show_XXXX'). Classifier: classify_pattern(deal_type, notes).


<a id="s-overview"></a>

## 1. Bird's-eye view: The Crescent's 24 months in the data

### Biggest takeaway

- **24 months, 501 past shows.** A complete operating record, not a sample. Patterns at 5%+ have 25+ shows behind them; patterns at 1% have 5+.
- **5 agencies, 14 agents, 59 artists.** Small enough that per-agent clustering is meaningful; artist-level is too long-tail.
- **501 deals, 501 settlements.** No orphans, no duplicates — denominators don't drift between sections.
- **Expenses and comps go deep** (2,769 and 1,817 line items). ~5.5 expense rows per show — that's why Wednesday is an aggregation problem.
- **36 future shows** already booked; out of scope (every dispute / settlement query filters `date <= 2026-05-18`).

Every percentage in the brief is `<some count> / 501`, `/ 14`, or `/ 52`. The next sections drill into each numerator.

[↓ View SQL for this section](#sql-overview)

In [2]:
q(OVERVIEW_SQL, {"gate": ANALYSIS_DATE})

,what,value
0,Time horizon (past shows),2024-05-18 to 2026-05-16
1,Past shows,501
2,"Future shows (booked, not yet happened)",36
3,Venue (single),The Crescent
4,Distinct artists (past shows),59
5,Distinct agents (past shows),14
6,Distinct agencies (past shows),5
7,Deals (one per show),501
8,Settlements (one per show),501
9,Expense line items,2769


<a id="s-dealmix"></a>

## 2. 316 of 501 deals (63%) are tool-unsupported

### Biggest takeaway

- **501 deals split into 5 `deal_type` values stored in `deals.deal_type`:** `vs` (175), `flat` (173), `percentage_of_net` (112), `door` (29), `percentage_of_gross` (12).
- **316 of those 501 are tool-unsupported.** Proof: `total_to_artist` matches the one-line formula on the deal terms in 185 / 185 cases for `flat` and `percentage_of_gross`, and 0 / 316 cases for the other three. The DB itself draws the line.

[↓ View SQL for this section](#sql-dealmix)

In [3]:
q(DEALMIX_SQL, {"gate": ANALYSIS_DATE})

,deal_type,n,pct_of_501,formula_match
0,vs,175,34.9,0
1,flat,173,34.5,173
2,percentage_of_net,112,22.4,0
3,door,29,5.8,0
4,percentage_of_gross,12,2.4,12


<a id="s-disputed"></a>

## 3. The 20 disputed settlements: who, what, where

### Biggest takeaway

- **20 settlements carry `status='disputed'`** — same number the in-app Reports tab shows. This is the conservative dispute count, and it's what §3 dissects.
- **19 of 20 sit on tool-unsupported deal types** (12 `vs`, 6 `percentage_of_net`, 1 `door`). The lone exception is `flat` (show_0142, Kev Park). Disputes track the deal types the calculator can't compute.
- **Clustering is per-agent, not per-agency.** Tom Neary (Wasserman) has 4 of 20; Cass Burke (Independent) has 3. The top 2 agents = 7 of 20 (35%) — backs the per-agent thesis in the brief.
- **19 of 20 disputed settlements carry a positive signoff** (*"Looks good"*, *"OK. Good night."*, *"👍"*, *"ok wire monday"*). The badge contradicts the prose. Only `show_coastal_spell_dispute` has a signoff that flags the issue.
- **Only 1 of 20 has a disputed recoup line item inside `recoups_json`** (coastal_spell). The other 19 disputes don't show their cause at the recoup level — the dispute is in the deal math itself, not in a recoup challenge.

[↓ View SQL for this section](#sql-disputed)

In [4]:
# 3a. The 20 disputed shows, one row per settlement.
q(DISPUTED_DRILL_SQL, {"gate": ANALYSIS_DATE})

,show_id,date,deal_type,agent,agency,guar,payout,recoup_disputed,signoff_snippet
0,show_0007,2024-10-24,vs,Rosa Jimenez,Wasserman,2631.0,7043.6,.,Looks good — TM. Wire to the usual account when ready.
1,show_0015,2025-12-02,vs,Kev Park,Paradigm,2449.0,4697.0,.,OK. Good night.
2,show_0036,2025-09-20,door,Tom Neary,Wasserman,NaN,2304.0,.,Looks good.
3,show_0044,2024-09-14,percentage_of_net,Tom Neary,Wasserman,NaN,7692.5,.,OK. Good night.
4,show_0064,2025-02-21,vs,Jordan Wells,Independent,792.0,792.0,.,👍
5,show_0069,2025-06-04,vs,Jordan Wells,Independent,812.0,2323.2,.,👍
6,show_0142,2024-08-30,flat,Kev Park,Paradigm,1208.0,1208.0,.,ok wire monday
7,show_0152,2025-03-22,vs,Tom Neary,Wasserman,4988.0,14641.2,.,Looks good.
8,show_0182,2026-05-08,vs,Pat Cho,Independent,1275.0,4129.5,.,OK. Good night.
9,show_0226,2026-01-31,vs,Cass Burke,Independent,1578.0,4605.3,.,Looks good.


In [5]:
# 3b. Rollups: same 20 disputes, grouped by deal_type and by agent.
print("Disputes by deal_type")
display(q(DISPUTED_BY_DEALTYPE_SQL, {"gate": ANALYSIS_DATE}))
print("\nDisputes by agent")
display(q(DISPUTED_BY_AGENT_SQL, {"gate": ANALYSIS_DATE}))

Disputes by deal_type


,deal_type,n_disputed
0,vs,12
1,percentage_of_net,6
2,flat,1
3,door,1



Disputes by agent


,agent,agency,n_disputed
0,Tom Neary,Wasserman,4
1,Cass Burke,Independent,3
2,Danny Ortiz,CAA,2
3,Jordan Wells,Independent,2
4,Kev Park,Paradigm,2
5,Maya Okafor,Independent,2
6,Rosa Jimenez,Wasserman,2
7,Daniel Hwang,WME,1
8,Naomi Brand,Paradigm,1
9,Pat Cho,Independent,1


<a id="s-recoup-disputed"></a>

## 4. Hidden disputes inside `recoups_json`: the UI undercounts disputes 2×

### Biggest takeaway

- **The UI shows 20 disputes (`status='disputed'`). The DB has 22 more, hiding inside `recoups_json` on settlements that look clean.** A recoup line item carries its own `status` field; when it equals `"disputed"`, the artist or TM rejected a specific recoup but the overall settlement was marked `paid` or `finalized` and shipped anyway.
- **23 settlements have at least one disputed recoup line item.** Of those, **21 are marked `paid`**, 1 is `finalized`, and 1 is the coastal_spell case already in §3. So §4 surfaces 22 disputes the UI never flags.
- **Combined dispute footprint: 20 (§3) + 22 (§4) = 42 settlements (8.4% of 501)** — more than twice the 4.0% the in-app Reports tab claims.
- **Categories cluster around marketing recoups** (12 line items, $5,725 — the largest category by both count and dollars), production overage (8 items, $1,981), and hospitality overage (4 items, $377). 4 of the 12 marketing line items share the `hwang_pattern` id signature: Daniel Hwang (WME) pushes back on marketing recoups post-show as a repeated behavior.

[↓ View SQL for this section](#sql-recoup-disputed)

In [6]:
# 4a. Every disputed recoup line item, one row per item.
q(RECOUP_DISPUTED_DRILL_SQL, {"gate": ANALYSIS_DATE})

,show_id,date,settlement_status,deal_type,agent,agency,category,label,amount,signoff_snippet
0,show_0000,2026-04-14,paid,flat,Cass Burke,Independent,production_overage,Backline: drum riser,219,👍
1,show_0002,2024-09-09,paid,vs,Kev Park,Paradigm,hospitality_overage,Over $400 hospitality cap,77,ok wire monday
2,show_0004,2024-11-13,paid,flat,Daniel Hwang,WME,marketing,Marketing recoup (post-show pushback),681,Looks good.
3,show_0005,2024-09-05,paid,vs,Tom Neary,Wasserman,hospitality_overage,Over $300 hospitality cap,70,Looks good.
4,show_0011,2024-09-27,paid,vs,Daniel Hwang,WME,marketing,Marketing recoup (post-show pushback),808,ok wire monday
5,show_0023,2025-05-22,paid,vs,Daniel Hwang,WME,marketing,Marketing recoup (post-show pushback),762,👍
6,show_0035,2024-11-07,paid,vs,Daniel Hwang,WME,marketing,Marketing recoup (post-show pushback),759,ok wire monday
7,show_0076,2025-04-08,paid,percentage_of_net,Jordan Wells,Independent,hospitality_overage,Over $400 hospitality cap,129,👍
8,show_0153,2025-10-17,paid,door,Jordan Wells,Independent,marketing,Pre-show ad spend,469,Looks good.
9,show_0170,2024-11-04,paid,percentage_of_net,Daniel Hwang,WME,marketing,Marketing recoup (post-show pushback),388,OK. Good night.


In [7]:
# 4b. Disputed recoup line items rolled up by category, with dollar totals.
q(RECOUP_DISPUTED_BY_CATEGORY_SQL, {"gate": ANALYSIS_DATE})

,category,n_line_items,n_settlements,total_disputed_amount
0,marketing,12,12,5725
1,production_overage,8,8,1981
2,hospitality_overage,4,4,377


<a id="s-clustering"></a>

## 5. Per-agent and per-agency clustering on the full 42-dispute universe

### Biggest takeaway

- **Clustering is per-agent, and the top of the list looks different once hidden disputes are counted.** On the 42-universe (20 UI-visible + 22 hidden in `recoups_json`): **Cass Burke and Daniel Hwang tie at 6 each**, Jordan Wells and Tom Neary at 5, Maya Okafor and Rosa Jimenez at 4. Top 6 agents = 30 of 42 = **71%**.
- **Daniel Hwang is the case the UI hides.** UI-visible: 1 dispute. Hidden recoup-level: 5. Total: 6. The hidden 5 are the `hwang_pattern` post-show marketing-recoup pushbacks against 5 different shows. Without §4 he looks like a low-volume agent. With §4 he's tied for the top cluster.
- **Per-agency reorders:** Independent **18 (43%)**, Wasserman **9 (21%)**, WME **7 (17%)**, Paradigm 4 (10%), CAA 4 (10%). Independent overtakes Wasserman as the largest agency bucket because Cass Burke + Jordan Wells + Maya Okafor + Pat Cho all sit there. WME rises from "barely above base rate" (UI-only view) to top-three once Hwang's pattern is included.
- **Mariana's "all WME" gut is more accurate than the UI suggests.** The UI undercounts WME's dispute exposure because it doesn't show Hwang's marketing-recoup pushback. Her recall is reading a signal the in-app Reports tab structurally misses.

[↓ View SQL for this section](#sql-clustering)

In [8]:
# 5a. Per-agent rollup on the 42-dispute universe.
# n_ui_visible + n_hidden_only = n_total. Sorted by n_total desc.
q(DISPUTE_UNIVERSE_BY_AGENT_SQL, {"gate": ANALYSIS_DATE})

,agent,agency,n_ui_visible,n_hidden_only,n_total
0,Cass Burke,Independent,3,3,6
1,Daniel Hwang,WME,1,5,6
2,Jordan Wells,Independent,2,3,5
3,Tom Neary,Wasserman,4,1,5
4,Maya Okafor,Independent,2,2,4
5,Rosa Jimenez,Wasserman,2,2,4
6,Kev Park,Paradigm,2,1,3
7,Pat Cho,Independent,1,2,3
8,Danny Ortiz,CAA,2,0,2
9,Meera Patel,CAA,0,2,2


In [9]:
# 5b. Per-agency rollup on the 42-dispute universe, with % of 42.
q(DISPUTE_UNIVERSE_BY_AGENCY_SQL, {"gate": ANALYSIS_DATE})

,agency,n_ui_visible,n_hidden_only,n_total,pct_of_42
0,Independent,8,10,18,42.9
1,Wasserman,6,3,9,21.4
2,WME,1,6,7,16.7
3,Paradigm,3,1,4,9.5
4,CAA,2,2,4,9.5


<a id="s-patterns"></a>

## 6. 501 deals collapse into 12 patterns (and 1 unmatched bucket)

### Biggest takeaway

- **The data is not 501 bespoke shapes. It is 12.** Excluding the 173 flat deals (already calculator-supported per §2), the other 328 deals across `vs`, `% of net`, `door`, and `% of gross` partition into exactly 12 patterns with 0 unmatched.
- **The 9 Vs sub-variants are the schema-gap detail.** `deals.deal_type='vs'` is one column with one value, but the prose splits those 175 rows into nine mechanically-distinct shapes — slash-splits with and without walkout, escalator/ratchet, vs-gross with and without risk caveat, walkout pot spelled out, inline capacity ratchet, compact and "whichever greater" forms. Any structured deal model has to hold all nine.
- **Pattern coverage is the deal-modeling thesis turned into a verifiable claim.** If 328 free-text entries fit 12 regex shapes with no outliers, then "model the data into a structured schema" is a tractable scope, not an open-ended NLP project.
- **The regexes are inspectable** in the Setup cell (`VS_PATTERNS`). A leader who doubts the partition can read each pattern, modify it, rerun, and see if the counts change. The classifier is auditable, not opaque.

[↓ View pattern definitions and SQL for this section](#sql-patterns)

In [10]:
# 6. Classify every non-flat deal into one of 12 patterns + Unmatched.
# Returns one row per pattern with count and 3 sample show_ids.
df = q(DEALS_FREETEXT_SQL, {"gate": ANALYSIS_DATE})
df["pattern_id"] = df.apply(
    lambda r: classify_pattern(r.deal_type, r.deal_notes_freetext), axis=1
)
# Drop flat (excluded by design).
classified = df[df.pattern_id.notna()].copy()

summary = (
    classified.groupby("pattern_id")
              .agg(n=("show_id", "count"),
                   example_show_ids=("show_id", lambda s: ", ".join(sorted(s)[:3])))
              .reindex(PATTERN_ORDER)
              .dropna(subset=["n"])
              .reset_index()
)
summary["description"] = summary["pattern_id"].map(PATTERN_DESCRIPTIONS)
summary["n"] = summary["n"].astype(int)
summary = summary[["pattern_id", "description", "n", "example_show_ids"]]

print(f"Classified: {summary['n'].sum()} of 328 non-flat deals "
      f"({len(classified)} rows, 0 unmatched expected)")
summary

Classified: 328 of 328 non-flat deals (328 rows, 0 unmatched expected)


,pattern_id,description,n,example_show_ids
0,V1,"Vs % of net, whichever greater",40,"show_0002, show_0063, show_0064"
1,V2,Vs slash-split (no walkout),36,"show_0005, show_0035, show_0052"
2,V3,Vs slash-split with walkout above breakeven,18,"show_0009, show_0011, show_0069"
3,V4,Vs % of net (compact),25,"show_0022, show_0023, show_0071"
4,V5,Vs escalator/ratchet over capacity,12,"show_0014, show_0053, show_0106"
5,V6,"Vs % of GROSS, whichever greater",9,"show_0016, show_0034, show_0066"
6,V7,Vs % of gross (with risk caveat),8,"show_0013, show_0047, show_0098"
7,V8,Vs + walkout pot,14,"show_0007, show_0015, show_0045"
8,V9,Vs inline capacity ratchet,13,"show_0072, show_0078, show_0134"
9,N1,"% of net, no guarantee",112,"show_0020, show_0021, show_0029"


<a id="s-email-thread"></a>

## 7. 19 deals live outside Greenroom: the scope floor for structured capture

### Biggest takeaway

- **3.8% of deals have a source of truth that is not in the Greenroom database.** 19 of 501 deals carry the literal phrase *"Performance bonuses per the deal memo (see email thread)"* in `deal_notes_freetext`. The bonus terms are too complex to fit in the structured fields *and* too complex to fit in the prose field — Mariana writes "see email thread" because the answer lives in her inbox, not in the system.
- **This is the floor for what structured capture can solve.** No deal model, however careful, will eliminate these. They are the irreducible "lives outside Greenroom" residue. The prototype's honest success criterion is ~96% structural coverage, not 100%.
- **The prototype must mark these explicitly.** A `bonuses_in_email_thread = true` flag on the deal record lets the calculator defer at settlement time ("bonuses not computable — check email") rather than silently producing the wrong payout. The schema doesn't need to parse the email; it needs to know the email is the source.
- **All 19 are `vs` deals.** None are flat, % of net, door, or % of gross. Complex bonus structure is a Vs-deal phenomenon (typically tiered "+$X if gross > $Y" stacks that exceed what prose can comfortably hold).

[↓ View SQL for this section](#sql-email-thread)

In [11]:
# 7. The 19 "see email thread" deals — every show with deal_type, agent, settlement status,
# and a snippet showing the literal phrase. Reader can scan agent column for counterparty mix.
q(SEE_EMAIL_THREAD_SQL, {"gate": ANALYSIS_DATE})

,show_id,date,deal_type,agent,agency,settlement_status,guar,expense_cap,deal_notes_snippet
0,show_0002,2024-09-09,vs,Kev Park,Paradigm,paid,1405.0,700.0,"$1,405 guarantee vs 90% of net after expenses, whichever greater. Expenses capped $700. Hospitality cap $400. Perfor..."
1,show_0013,2025-05-31,vs,Naomi Brand,Paradigm,paid,1868.0,NaN,"1,868 g'tee vs 85% gross — no expenses come out. Simpler math, riskier for venue. Hosp $500. Performance bonuses per..."
2,show_0022,2025-12-20,vs,Tom Neary,Wasserman,paid,5433.0,2700.0,"5,433 g'tee vs 80% of net. Expenses to 2700. Hospitality $500. Performance bonuses per the deal memo (see email thre..."
3,show_0064,2025-02-21,vs,Jordan Wells,Independent,disputed,792.0,400.0,"$792 guarantee vs 85% of net after expenses, whichever greater. Expenses capped $400. Hospitality cap $500. Performa..."
4,show_0070,2025-02-07,vs,Maya Okafor,Independent,paid,1763.0,900.0,"$1,763 guarantee vs 70% of net after expenses, whichever greater. Expenses capped $900. Hospitality cap $300. Perfor..."
5,show_0071,2025-01-22,vs,Chris Lockhart,CAA,paid,1534.0,750.0,"1,534 g'tee vs 70% of net. Expenses to 750. Hospitality $500. Performance bonuses per the deal memo (see email thread)."
6,show_0094,2025-12-10,vs,Maya Okafor,Independent,paid,1496.0,750.0,"$1,496 guarantee vs 85% of net after expenses, whichever greater. Expenses capped $750. Hospitality cap $600. Perfor..."
7,show_0097,2025-01-21,vs,Kev Park,Paradigm,paid,5151.0,2600.0,"$5,151 guarantee vs 90% of net after expenses, whichever greater. Expenses capped $2600. Hospitality cap $300. Perfo..."
8,show_0186,2024-11-12,vs,Cass Burke,Independent,paid,817.0,NaN,"$817 vs 90% of GROSS (no expense deductions), whichever greater. Hospitality cap $500. Performance bonuses per the d..."
9,show_0191,2025-08-08,vs,Andrea Pelletier,WME,paid,2527.0,NaN,"$2,527 vs 90% of GROSS (no expense deductions), whichever greater. Hospitality cap $300. Performance bonuses per the..."


<a id="s-coastal-spell"></a>

## 8. Coastal Spell: every pattern at scale, in one row

### Biggest takeaway

This is the canonical case study — the show all five interview participants reference unprompted. Marcus pegs the routing-revenue consequence at ~$80K (one bad relationship that didn't return for the next tour). The DB earns its mention because **every dispute pattern in §3-§5 intersects on this single row**:

- **§3 ∩ §4: the one show with both signals.** Every other dispute in the corpus is either `status='disputed'` OR has a `recoups_json` line item flagged disputed — never both. Coastal Spell is the only row that hits both surfaces. The badge is `disputed` AND there's a $900 marketing-recoup line item marked `"status":"disputed"`.
- **FK-vs-prose drift, in one row.** The structured foreign keys say the counterparty is **Tom Neary at Wasserman**. The prose in `deal_notes_freetext` reads *"recoup interpretation disputed by WME"* — the database disagrees with itself, in the same row, across structured and unstructured fields. Any shared-deal-artifact prototype has to decide which side wins.
- **The dispute is recoup-axis, not deal-math.** The vs % of net math worked. The disagreement was *"is the $900 Spotify pre-show ad spend a recoupable marketing expense against gross?"* — the same recoup-eligibility ambiguity the §4 finding describes at scale, but here it surfaced as a badge dispute instead of hiding inside a `paid` settlement.
- **90 days of latency from email to surfacing, resolved with a $720 concession.** Deal email arrived December 2024; show was 2025-03-14; the dispute resolved 2025-03-19 with $720 returned. Small dollar number, large trust cost — Marcus's $80K-of-routing-revenue testimony is the second-order consequence the DB cannot show but everyone in the building remembers.
- **The signoff is different from the other 19.** Every other UI-visible dispute in §3 has a generic positive signoff ("Looks good", "👍"). Coastal Spell's signoff reads *"OK — but flag any future marketing recoup deals."* Marcus is asking the system to remember this for next time. The system doesn't.

The Loom should park here for a minute. Every pattern this brief argues about can be pointed at on this single row.

[↓ View full record below](#s-coastal-spell-code)

In [12]:
# 8. Full record for Coastal Spell — every field on the deal + settlement, transposed.
# The disputed recoup line item is inside `recoups_json`; the prose-vs-FK drift is
# visible by comparing the structured agent/agency joins with `deal_notes_freetext`.
inspect("show_coastal_spell_dispute")

,0
show_id,show_coastal_spell_dispute
deal_type,vs
guarantee_amount,5000.0
percentage,0.8
percentage_basis,net
expense_cap,2500.0
hospitality_cap,500.0
deal_notes_freetext,"$5,000 vs 80% of net after expenses, whichever greater. Expenses capped $2,500. Hospitality cap $500. +$1,000 bonus ..."
status,disputed
gross_box_office,19840.0


<a id="sql-reference"></a>

---

# SQL Queries Reference

Every query used above, rendered as a formatted SQL code block. The strings here are the *same Python variables* defined in Setup — they cannot drift from what was executed.

In [13]:
pattern_md = "\n".join(
    f"| `{pid}` | {desc} | `{regex}` |" for pid, desc, regex in VS_PATTERNS
)
nonvs_md = "\n".join(
    f"| `{pid}` | {desc} | `deals.deal_type = '{dt}'` |"
    for dt, (pid, desc) in NON_VS_BUCKETS.items()
)

Markdown(f"""
<a id="sql-overview"></a>
### §1 — Bird's-eye view of the corpus

Counts each "shape" of the past-show universe (shows, deals, settlements, distinct artists/agents/agencies, expense/comp/ticket-sales rows) at the analysis boundary. Anchors every denominator that follows. [↑ Back to section](#s-overview)

```sql
{OVERVIEW_SQL.strip()}
```

<a id="sql-dealmix"></a>
### §2 — Deal mix and formula match

One row per `deal_type`. `n` and `pct_of_501` are the straight group-by. `formula_match` counts settlements where `total_to_artist` equals the one-line formula on the deal terms (`flat → guarantee_amount`, `percentage_of_gross → gross_box_office × percentage`). The other three types have no closed-form equivalent, so they read 0 by construction. [↑ Back to section](#s-dealmix)

```sql
{DEALMIX_SQL.strip()}
```

<a id="sql-disputed"></a>
### §3 — Disputed settlements: drilldown + rollups

**3a. Drilldown.** Every settlement with `status='disputed'`, joined to deals + artists + agents + agencies. `recoup_disputed` flags whether `recoups_json` contains a `"status":"disputed"` line item; `signoff_snippet` is the first 60 chars of `signoff_text`. [↑ Back to section](#s-disputed)

```sql
{DISPUTED_DRILL_SQL.strip()}
```

**3b. By deal_type.** Same 20 settlements, counted per `deal_type`.

```sql
{DISPUTED_BY_DEALTYPE_SQL.strip()}
```

**3b. By agent.** Same 20 settlements, counted per agent (with agency for context).

```sql
{DISPUTED_BY_AGENT_SQL.strip()}
```

<a id="sql-recoup-disputed"></a>
### §4 — Hidden disputes inside `recoups_json`

**4a. Drilldown.** Uses SQLite's `json_each` to flatten `settlements.recoups_json` into one row per line item, then filters for `status='disputed'`. Joins to deal + agent so each hidden dispute is attributable to a counterparty. [↑ Back to section](#s-recoup-disputed)

```sql
{RECOUP_DISPUTED_DRILL_SQL.strip()}
```

**4b. Category rollup.** Same line items grouped by recoup `category`, with line-item count, distinct-settlement count, and total dollar value of disputed recoups.

```sql
{RECOUP_DISPUTED_BY_CATEGORY_SQL.strip()}
```

<a id="sql-clustering"></a>
### §5 — Dispute clustering on the 42-universe (UI + hidden)

A `WITH dispute_universe` CTE selects every distinct `show_id` that has either `status='disputed'` (UI-visible) or at least one `recoups_json` line item with `status='disputed'` (hidden). Each row flags `is_ui_visible` and `has_hidden_recoup`. The two rollups split each agent / agency into `n_ui_visible` vs `n_hidden_only` so the gap between the in-app Reports tab and the DB truth is explicit. [↑ Back to section](#s-clustering)

**5a. Per-agent.**

```sql
{DISPUTE_UNIVERSE_BY_AGENT_SQL.strip()}
```

**5b. Per-agency.**

```sql
{DISPUTE_UNIVERSE_BY_AGENCY_SQL.strip()}
```

<a id="sql-patterns"></a>
### §6 — Pattern classifier

The 175 `vs` deals are classified by Python regex into 9 sub-variants. The other three non-flat deal types (`percentage_of_net`, `percentage_of_gross`, `door`) are bucketed by their `deal_type` enum value directly — no regex needed. Flat deals are excluded from the §6 summary because the in-app calculator already handles them per §2. [↑ Back to section](#s-patterns)

**Source query (pull every non-flat deal with its notes):**

```sql
{DEALS_FREETEXT_SQL.strip()}
```

**Vs sub-variant regexes** (priority order; first match wins):

| Pattern | Description | Regex |
|---|---|---|
{pattern_md}

**Non-Vs buckets** (no regex; uses `deal_type` enum value):

| Pattern | Description | Source |
|---|---|---|
{nonvs_md}

<a id="sql-email-thread"></a>
### §7 — "See email thread" deals (scope floor)

Single `LIKE` on `deals.deal_notes_freetext`. Returns every deal whose bonus terms punt to an external email thread — the irreducible residue of "lives outside Greenroom" that structured capture cannot eliminate. [↑ Back to section](#s-email-thread)

```sql
{SEE_EMAIL_THREAD_SQL.strip()}
```

<a id="s-coastal-spell-code"></a>
### §8 — Coastal Spell case study

Uses the `inspect()` helper defined in Setup. Pulls every column on the `deals` and `settlements` rows for `show_coastal_spell_dispute`, transposed for reading. The disputed recoup line item lives inside `recoups_json`; the prose-vs-FK drift is visible by comparing the structured `agent` / `agency` joins with the text of `deal_notes_freetext`. [↑ Back to section](#s-coastal-spell)

```python
inspect("show_coastal_spell_dispute")

# `inspect` is defined in the Setup cell as:
#
#   SELECT d.show_id, d.deal_type, d.guarantee_amount, d.percentage,
#          d.percentage_basis, d.expense_cap, d.hospitality_cap,
#          d.deal_notes_freetext,
#          st.status, st.gross_box_office, st.net_box_office,
#          st.total_expenses, st.total_to_artist,
#          st.calculation_json, st.signoff_text, st.notes AS settlement_notes
#     FROM deals d
#     JOIN settlements st ON d.show_id = st.show_id
#    WHERE d.show_id = :sid
```
""")


<a id="sql-overview"></a>
### §1 — Bird's-eye view of the corpus

Counts each "shape" of the past-show universe (shows, deals, settlements, distinct artists/agents/agencies, expense/comp/ticket-sales rows) at the analysis boundary. Anchors every denominator that follows. [↑ Back to section](#s-overview)

```sql
SELECT 'Time horizon (past shows)'         AS what,
       MIN(date) || '  to  ' || MAX(date)  AS value
  FROM shows WHERE date <= :gate
UNION ALL SELECT 'Past shows',
       CAST(COUNT(*) AS TEXT) FROM shows WHERE date <= :gate
UNION ALL SELECT 'Future shows (booked, not yet happened)',
       CAST(COUNT(*) AS TEXT) FROM shows WHERE date > :gate
UNION ALL SELECT 'Venue (single)',
       (SELECT name FROM venues LIMIT 1)
UNION ALL SELECT 'Distinct artists (past shows)',
       CAST(COUNT(DISTINCT artist_id) AS TEXT) FROM shows WHERE date <= :gate
UNION ALL SELECT 'Distinct agents (past shows)',
       CAST(COUNT(DISTINCT a.id) AS TEXT)
  FROM shows s
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  a  ON ar.agent_id = a.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Distinct agencies (past shows)',
       CAST(COUNT(DISTINCT a.agency_id) AS TEXT)
  FROM shows s
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  a  ON ar.agent_id = a.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Deals (one per show)',
       CAST(COUNT(*) AS TEXT) FROM deals d
  JOIN shows s ON d.show_id = s.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Settlements (one per show)',
       CAST(COUNT(*) AS TEXT) FROM settlements st
  JOIN shows s ON st.show_id = s.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Expense line items',
       CAST(COUNT(*) AS TEXT) FROM expenses e
  JOIN shows s ON e.show_id = s.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Comp line items',
       CAST(COUNT(*) AS TEXT) FROM comps c
  JOIN shows s ON c.show_id = s.id
 WHERE s.date <= :gate
UNION ALL SELECT 'Ticket-sales rows (one aggregate per show)',
       CAST(COUNT(*) AS TEXT) FROM ticket_sales t
  JOIN shows s ON t.show_id = s.id
 WHERE s.date <= :gate
```

<a id="sql-dealmix"></a>
### §2 — Deal mix and formula match

One row per `deal_type`. `n` and `pct_of_501` are the straight group-by. `formula_match` counts settlements where `total_to_artist` equals the one-line formula on the deal terms (`flat → guarantee_amount`, `percentage_of_gross → gross_box_office × percentage`). The other three types have no closed-form equivalent, so they read 0 by construction. [↑ Back to section](#s-dealmix)

```sql
SELECT d.deal_type,
       COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) /
             (SELECT COUNT(*) FROM shows WHERE date <= :gate), 1) AS pct_of_501,
       SUM(CASE
             WHEN d.deal_type = 'flat'
                  AND ABS(st.total_to_artist - d.guarantee_amount) < 0.5
                  THEN 1
             WHEN d.deal_type = 'percentage_of_gross'
                  AND ABS(st.total_to_artist - (st.gross_box_office * d.percentage)) < 1
                  THEN 1
             ELSE 0
           END) AS formula_match
  FROM deals d
  JOIN shows s ON d.show_id = s.id
  JOIN settlements st ON st.show_id = s.id
 WHERE s.date <= :gate
 GROUP BY d.deal_type
 ORDER BY n DESC
```

<a id="sql-disputed"></a>
### §3 — Disputed settlements: drilldown + rollups

**3a. Drilldown.** Every settlement with `status='disputed'`, joined to deals + artists + agents + agencies. `recoup_disputed` flags whether `recoups_json` contains a `"status":"disputed"` line item; `signoff_snippet` is the first 60 chars of `signoff_text`. [↑ Back to section](#s-disputed)

```sql
SELECT s.id AS show_id,
       s.date,
       d.deal_type,
       ag.name AS agent,
       agcy.name AS agency,
       d.guarantee_amount AS guar,
       st.total_to_artist AS payout,
       CASE WHEN st.recoups_json LIKE '%"status":"disputed"%'
            THEN 'Y' ELSE '.' END AS recoup_disputed,
       substr(st.signoff_text, 1, 60) AS signoff_snippet
  FROM settlements st
  JOIN shows s   ON st.show_id  = s.id
  JOIN deals d   ON d.show_id   = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id
 WHERE s.date <= :gate
   AND st.status = 'disputed'
 ORDER BY s.id
```

**3b. By deal_type.** Same 20 settlements, counted per `deal_type`.

```sql
SELECT d.deal_type,
       COUNT(*) AS n_disputed
  FROM settlements st
  JOIN shows s ON st.show_id = s.id
  JOIN deals d ON d.show_id  = s.id
 WHERE s.date <= :gate
   AND st.status = 'disputed'
 GROUP BY d.deal_type
 ORDER BY n_disputed DESC
```

**3b. By agent.** Same 20 settlements, counted per agent (with agency for context).

```sql
SELECT ag.name AS agent,
       agcy.name AS agency,
       COUNT(*) AS n_disputed
  FROM settlements st
  JOIN shows s ON st.show_id = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id
 WHERE s.date <= :gate
   AND st.status = 'disputed'
 GROUP BY ag.id
 ORDER BY n_disputed DESC, agent
```

<a id="sql-recoup-disputed"></a>
### §4 — Hidden disputes inside `recoups_json`

**4a. Drilldown.** Uses SQLite's `json_each` to flatten `settlements.recoups_json` into one row per line item, then filters for `status='disputed'`. Joins to deal + agent so each hidden dispute is attributable to a counterparty. [↑ Back to section](#s-recoup-disputed)

```sql
SELECT s.id AS show_id,
       s.date,
       st.status AS settlement_status,
       d.deal_type,
       ag.name AS agent,
       agcy.name AS agency,
       json_extract(je.value, '$.category') AS category,
       json_extract(je.value, '$.label')    AS label,
       json_extract(je.value, '$.amount')   AS amount,
       substr(st.signoff_text, 1, 60)       AS signoff_snippet
  FROM settlements st
  JOIN shows s   ON st.show_id  = s.id
  JOIN deals d   ON d.show_id   = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id,
  json_each(st.recoups_json) je
 WHERE s.date <= :gate
   AND json_extract(je.value, '$.status') = 'disputed'
 ORDER BY s.id, json_extract(je.value, '$.label')
```

**4b. Category rollup.** Same line items grouped by recoup `category`, with line-item count, distinct-settlement count, and total dollar value of disputed recoups.

```sql
SELECT json_extract(je.value, '$.category') AS category,
       COUNT(*) AS n_line_items,
       COUNT(DISTINCT st.show_id) AS n_settlements,
       SUM(json_extract(je.value, '$.amount')) AS total_disputed_amount
  FROM settlements st
  JOIN shows s ON st.show_id = s.id,
  json_each(st.recoups_json) je
 WHERE s.date <= :gate
   AND json_extract(je.value, '$.status') = 'disputed'
 GROUP BY category
 ORDER BY n_line_items DESC
```

<a id="sql-clustering"></a>
### §5 — Dispute clustering on the 42-universe (UI + hidden)

A `WITH dispute_universe` CTE selects every distinct `show_id` that has either `status='disputed'` (UI-visible) or at least one `recoups_json` line item with `status='disputed'` (hidden). Each row flags `is_ui_visible` and `has_hidden_recoup`. The two rollups split each agent / agency into `n_ui_visible` vs `n_hidden_only` so the gap between the in-app Reports tab and the DB truth is explicit. [↑ Back to section](#s-clustering)

**5a. Per-agent.**

```sql
WITH dispute_universe AS (
  SELECT DISTINCT
         st.show_id,
         CASE WHEN st.status = 'disputed' THEN 1 ELSE 0 END AS is_ui_visible,
         CASE WHEN EXISTS (
                SELECT 1 FROM json_each(st.recoups_json) je
                 WHERE json_extract(je.value, '$.status') = 'disputed'
              ) THEN 1 ELSE 0 END AS has_hidden_recoup
    FROM settlements st
    JOIN shows s ON st.show_id = s.id
   WHERE s.date <= :gate
     AND ( st.status = 'disputed'
        OR EXISTS (
             SELECT 1 FROM json_each(st.recoups_json) je
              WHERE json_extract(je.value, '$.status') = 'disputed'
           )
         )
)

SELECT ag.name AS agent,
       agcy.name AS agency,
       SUM(du.is_ui_visible) AS n_ui_visible,
       SUM(CASE WHEN du.is_ui_visible = 0 AND du.has_hidden_recoup = 1
                THEN 1 ELSE 0 END) AS n_hidden_only,
       COUNT(*) AS n_total
  FROM dispute_universe du
  JOIN shows s ON du.show_id = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id
 GROUP BY ag.id
 ORDER BY n_total DESC, agent
```

**5b. Per-agency.**

```sql
WITH dispute_universe AS (
  SELECT DISTINCT
         st.show_id,
         CASE WHEN st.status = 'disputed' THEN 1 ELSE 0 END AS is_ui_visible,
         CASE WHEN EXISTS (
                SELECT 1 FROM json_each(st.recoups_json) je
                 WHERE json_extract(je.value, '$.status') = 'disputed'
              ) THEN 1 ELSE 0 END AS has_hidden_recoup
    FROM settlements st
    JOIN shows s ON st.show_id = s.id
   WHERE s.date <= :gate
     AND ( st.status = 'disputed'
        OR EXISTS (
             SELECT 1 FROM json_each(st.recoups_json) je
              WHERE json_extract(je.value, '$.status') = 'disputed'
           )
         )
)

SELECT agcy.name AS agency,
       SUM(du.is_ui_visible) AS n_ui_visible,
       SUM(CASE WHEN du.is_ui_visible = 0 AND du.has_hidden_recoup = 1
                THEN 1 ELSE 0 END) AS n_hidden_only,
       COUNT(*) AS n_total,
       ROUND(100.0 * COUNT(*) / 42.0, 1) AS pct_of_42
  FROM dispute_universe du
  JOIN shows s ON du.show_id = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id
 GROUP BY agcy.id
 ORDER BY n_total DESC
```

<a id="sql-patterns"></a>
### §6 — Pattern classifier

The 175 `vs` deals are classified by Python regex into 9 sub-variants. The other three non-flat deal types (`percentage_of_net`, `percentage_of_gross`, `door`) are bucketed by their `deal_type` enum value directly — no regex needed. Flat deals are excluded from the §6 summary because the in-app calculator already handles them per §2. [↑ Back to section](#s-patterns)

**Source query (pull every non-flat deal with its notes):**

```sql
SELECT s.id AS show_id, d.deal_type, d.deal_notes_freetext
  FROM deals d JOIN shows s ON d.show_id = s.id
 WHERE s.date <= :gate
```

**Vs sub-variant regexes** (priority order; first match wins):

| Pattern | Description | Regex |
|---|---|---|
| `V8` | Vs + walkout pot | `walkout pot` |
| `V7` | Vs % of gross (with risk caveat) | `gross.*(?:Simpler math|riskier for venue)` |
| `V6` | Vs % of GROSS, whichever greater | `vs\s+\d+%\s+of\s+GROSS` |
| `V5` | Vs escalator/ratchet over capacity | `escalator|ratchets\s+to` |
| `V9` | Vs inline capacity ratchet | `\d+%\s+net\s+to\s+\d+%\s+sold` |
| `V3` | Vs slash-split with walkout above breakeven | `\d+/\d+\s+net,\s+walkout\s+above` |
| `V2` | Vs slash-split (no walkout) | `\d+/\d+\s+(?:net|after\s+expenses|split)` |
| `V1` | Vs % of net, whichever greater | `of\s+net\s+after\s+expenses.*whichever\s+greater` |
| `V4` | Vs % of net (compact) | `g'tee\s+vs\s+\d+%\s+of\s+net` |

**Non-Vs buckets** (no regex; uses `deal_type` enum value):

| Pattern | Description | Source |
|---|---|---|
| `N1` | % of net, no guarantee | `deals.deal_type = 'percentage_of_net'` |
| `G1` | % of gross | `deals.deal_type = 'percentage_of_gross'` |
| `D1` | Door deal | `deals.deal_type = 'door'` |

<a id="sql-email-thread"></a>
### §7 — "See email thread" deals (scope floor)

Single `LIKE` on `deals.deal_notes_freetext`. Returns every deal whose bonus terms punt to an external email thread — the irreducible residue of "lives outside Greenroom" that structured capture cannot eliminate. [↑ Back to section](#s-email-thread)

```sql
SELECT s.id AS show_id,
       s.date,
       d.deal_type,
       ag.name AS agent,
       agcy.name AS agency,
       st.status AS settlement_status,
       d.guarantee_amount AS guar,
       d.expense_cap,
       substr(d.deal_notes_freetext, 1, 130) AS deal_notes_snippet
  FROM deals d
  JOIN shows s ON d.show_id = s.id
  JOIN artists ar ON s.artist_id = ar.id
  JOIN agents  ag ON ar.agent_id = ag.id
  JOIN agencies agcy ON ag.agency_id = agcy.id
  JOIN settlements st ON st.show_id = s.id
 WHERE s.date <= :gate
   AND d.deal_notes_freetext LIKE '%see email thread%'
 ORDER BY s.id
```

<a id="s-coastal-spell-code"></a>
### §8 — Coastal Spell case study

Uses the `inspect()` helper defined in Setup. Pulls every column on the `deals` and `settlements` rows for `show_coastal_spell_dispute`, transposed for reading. The disputed recoup line item lives inside `recoups_json`; the prose-vs-FK drift is visible by comparing the structured `agent` / `agency` joins with the text of `deal_notes_freetext`. [↑ Back to section](#s-coastal-spell)

```python
inspect("show_coastal_spell_dispute")

# `inspect` is defined in the Setup cell as:
#
#   SELECT d.show_id, d.deal_type, d.guarantee_amount, d.percentage,
#          d.percentage_basis, d.expense_cap, d.hospitality_cap,
#          d.deal_notes_freetext,
#          st.status, st.gross_box_office, st.net_box_office,
#          st.total_expenses, st.total_to_artist,
#          st.calculation_json, st.signoff_text, st.notes AS settlement_notes
#     FROM deals d
#     JOIN settlements st ON d.show_id = st.show_id
#    WHERE d.show_id = :sid
```
